mount Drive, pull the PDFs

In [1]:
!pip install -q pymupdf

import os
import shutil
import subprocess
import zipfile
from pathlib import Path
import pandas as pd
import torch
from google.colab import drive

drive.mount("/content/drive")

CONFIG = {
    "project_dir": Path("/content/drive/MyDrive/doculens_lava"),
    "output_dir": Path("/content/drive/MyDrive/doculens_lava/outputs"),
    "data_dir": Path("/content/lava_data"),
    "train_pdf_dir": Path("/content/lava_data/train_pdfs/train_pdfs"),
    "test_pdf_dir": Path("/content/lava_data/test_pdfs/test_pdfs"),
    "num_pages": 3,
    "render_dpi": 150,
}

# Download PDFs to local Colab disk if this is a fresh runtime.
if not CONFIG["test_pdf_dir"].exists():
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    shutil.copy(CONFIG["project_dir"] / "kaggle.json", kaggle_dir / "kaggle.json")
    os.chmod(kaggle_dir / "kaggle.json", 0o600)

    CONFIG["data_dir"].mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["kaggle", "competitions", "download", "-c", "lava-challenge-2026", "-p", str(CONFIG["data_dir"])],
        check=True,
    )
    zip_path = CONFIG["data_dir"] / "lava-challenge-2026.zip"
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(CONFIG["data_dir"])
    zip_path.unlink()

pages_df = pd.read_parquet(CONFIG["output_dir"] / "pages.parquet")
train_df = pd.read_parquet(CONFIG["output_dir"] / "retrieved_train.parquet")
test_df = pd.read_parquet(CONFIG["output_dir"] / "retrieved_test.parquet")
sub_df = pd.read_csv(CONFIG["output_dir"] / "sample_submission.csv")

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
print("Hardware   :", device)
print("Pages text :", len(pages_df), "| Train:", len(train_df), "| Test:", len(test_df))
print("Train PDFs :", len(list(CONFIG["train_pdf_dir"].glob("*.pdf"))),
      "| Test PDFs:", len(list(CONFIG["test_pdf_dir"].glob("*.pdf"))))

Mounted at /content/drive
Hardware   : CPU only
Pages text : 4772 | Train: 16 | Test: 624
Train PDFs : 5 | Test PDFs: 200


Connect to Gemini API

In [9]:
!pip install -q google-genai

import os
from getpass import getpass
from google import genai

CONFIG["vlm_model"] = "gemini-3.5-flash-lite"

# Ask for your Gemini API key without saving it inside the notebook code.
#if not os.environ.get("GEMINI_API_KEY"):
os.environ["GEMINI_API_KEY"] = getpass("Paste your GEMINI_API_KEY: ")

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Quick 1-second ping to verify the key and model work.
ping = gemini_client.models.generate_content(
    model=CONFIG["vlm_model"],
    contents="Reply with only the word: OK",
)
print("Model:", CONFIG["vlm_model"], "| Connection test:", ping.text.strip())

Paste your GEMINI_API_KEY: ··········
Model: gemini-3.5-flash-lite | Connection test: OK


Define ask_vlm()

In [3]:
import json
import time
import pymupdf
from PIL import Image
from google.genai import types

# Build a fast lookup dictionary: (split, file_id, page_number) -> extracted text
PAGE_TEXT = {
    (row.split, str(row.file_id), int(row.page_number)): str(row.text)[:2500]
    for row in pages_df.itertuples(index=False)
}


def render_page_image(pdf_path, page_number, dpi=150):
    """Render one 1-based PDF page as a sharp RGB PIL Image."""
    doc = pymupdf.open(pdf_path)
    page = doc[int(page_number) - 1]
    pix = page.get_pixmap(dpi=dpi, colorspace=pymupdf.csRGB)
    doc.close()
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


def ask_vlm(pdf_path, file_id, page_numbers, question, answer_format, split="train"):
    """
    Send up to CONFIG['num_pages'] page images + extracted text to Gemini.
    Returns a dict: {'answer': str, 'evidence_page_number': list[int], 'reasoning': str}.
    """
    pages = [int(p) for p in page_numbers][:CONFIG["num_pages"]]

    prompt = (
        "You are a precise document QA system. Use BOTH the extracted text (for exact spelling, codes, numbers) "
        "and the page images (for charts, bars, tables, and layout).\n"
        f"Question: {question}\n"
        f"Required answer_format: {answer_format}\n\n"
        "Rules:\n"
        "1. Answer in the EXACT same language as the question.\n"
        "2. In 'reasoning', write 2-3 short lines showing the exact values or math.\n"
        "3. In 'answer':\n"
        "   - 'string': ONLY the exact short phrase/name/value from the document.\n"
        "   - 'number': ONLY the number; keep units like '%', '㎡/人', or '倍' if the question asks for a rate/percentage/unit.\n"
        "   - 'unordered_list' or 'ordered_list': a JSON array of exact verbatim items (numbers if pure numbers, otherwise strings).\n"
        "4. In 'evidence_page_number', return a JSON array of ONLY the integer page numbers you actually used.\n"
        "5. Return valid JSON with keys: 'reasoning', 'answer', 'evidence_page_number'."
    )

    contents = [prompt]
    for p in pages:
        text = PAGE_TEXT.get((split, str(file_id), p), "")
        img = render_page_image(pdf_path, p, dpi=CONFIG["render_dpi"])
        contents.append(f"--- Page {p} Extracted Text ---\n{text}")
        contents.append(img)

    for attempt in range(3):
        try:
            response = gemini_client.models.generate_content(
                model=CONFIG["vlm_model"],
                contents=contents,
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                ),
            )
            result = json.loads(response.text.strip())

            raw_ans = result.get("answer", "")
            answer = str(raw_ans) if isinstance(raw_ans, list) else str(raw_ans).strip()

            raw_pages = result.get("evidence_page_number", pages[:1])
            used_pages = sorted(set(int(x) for x in raw_pages if str(x).strip().isdigit()))
            if not used_pages:
                used_pages = pages[:1]

            return {
                "answer": answer,
                "evidence_page_number": used_pages,
                "reasoning": str(result.get("reasoning", "")),
            }
        except Exception as err:
            if attempt == 2:
                print(f"  [ERROR on {file_id}]: {err}")
                return {"answer": "", "evidence_page_number": pages[:1], "reasoning": str(err)}
            time.sleep(5 * (attempt + 1))


print("Cell 3 ready: ask_vlm() defined | Cached page texts:", len(PAGE_TEXT))

Cell 3 ready: ask_vlm() defined | Cached page texts: 4772


**test** it across all 16 train.csv questions.

In [4]:
import ast
import time

exact_ans = 0
exact_pages = 0
train_results = []
start_all = time.time()

print(f"{'ID':<7} | {'TIME':<5} | {'ANS':<5} | {'PG':<5} | TRUE -> PRED")
print("-" * 95)

for row in train_df.itertuples(index=False):
    pdf_path = CONFIG["train_pdf_dir"] / f"{row.file_id}.pdf"
    top_pages = [int(p) for p in row.top5_pages][:CONFIG["num_pages"]]
    true_pages = sorted(ast.literal_eval(str(row.evidence_page_number)))
    true_ans = str(row.answer).strip()

    t0 = time.time()
    pred = ask_vlm(
        pdf_path=pdf_path,
        file_id=row.file_id,
        page_numbers=top_pages,
        question=row.question,
        answer_format=row.answer_format,
        split="train",
    )
    dt = time.time() - t0

    ans_ok = (pred["answer"] == true_ans)
    pg_ok = (pred["evidence_page_number"] == true_pages)
    exact_ans += int(ans_ok)
    exact_pages += int(pg_ok)

    train_results.append({
        "id": row.id,
        "true_answer": true_ans,
        "pred_answer": pred["answer"],
        "true_pages": true_pages,
        "pred_pages": pred["evidence_page_number"],
        "reasoning": pred["reasoning"],
    })

    print(
        f"{row.id:<7} | {dt:>4.1f}s | {str(ans_ok):<5} | {str(pg_ok):<5} | "
        f"{true_ans} -> {pred['answer']} | pages {true_pages} -> {pred['evidence_page_number']}"
    )
    time.sleep(4)  # Stay cleanly under Gemini's free-tier requests-per-minute limit.

total_sec = time.time() - start_all
print("\n=== TRAIN VALIDATION SUMMARY (16 ROWS) ===")
print(f"Exact Answer Match : {exact_ans}/{len(train_df)} ({exact_ans / len(train_df) * 100:.1f}%)")
print(f"Exact Page Match   : {exact_pages}/{len(train_df)} ({exact_pages / len(train_df) * 100:.1f}%)")
print(f"Total time         : {total_sec:.1f}s | Avg: {total_sec / len(train_df):.1f}s/row")

ID      | TIME  | ANS   | PG    | TRUE -> PRED
-----------------------------------------------------------------------------------------------
q_0000  |  5.0s | True  | True  | ドイツ -> ドイツ | pages [11, 12] -> [11, 12]
q_0001  |  2.5s | True  | True  | 5% -> 5% | pages [1, 3] -> [1, 3]
q_0002  |  2.6s | True  | True  | 43 -> 43 | pages [14, 15] -> [14, 15]
q_0003  |  5.7s | False | True  | ['ITリテラシーの向上', '実践的なITスキルの習得'] -> ['IT企業への就職・起業', '実践的なITスキルの習得'] | pages [13, 21] -> [13, 21]
q_0004  |  2.6s | True  | True  | 在庫管理の向上 -> 在庫管理の向上 | pages [3, 9] -> [3, 9]
q_0005  |  4.0s | True  | False | 学習・生活空間の充実 -> 学習・生活空間の充実 | pages [2, 3, 4] -> [1, 2]
q_0006  |  4.1s | True  | False | ['袋井消防庁舎・袋井市防災センター整備事業', '袋井市静岡モデル防潮堤整備事業'] -> ['袋井消防庁舎・袋井市防災センター整備事業', '袋井市静岡モデル防潮堤整備事業'] | pages [14, 15] -> [13, 14]
q_0007  |  2.6s | True  | False | 経営企画・組織改革 -> 経営企画・組織改革 | pages [8] -> [2, 8]
q_0008  |  2.9s | True  | True  | 38.4㎡/人 -> 38.4㎡/人 | pages [1] -> [1]
q_0009  |  3.2s | False | True  | ['4NPhA', 

Run All 624 Test Rows (Checkpointed) & Export submission.csv

In [7]:
import time
import pandas as pd

# Unlock full Top-5 retrieval recall (98.4% on train) since Gemini has no 3-image cap.
CONFIG["num_pages"] = 5

ckpt_path = CONFIG["output_dir"] / "test_preds_checkpoint.parquet"
sub_path = CONFIG["output_dir"] / "submission.csv"

# Load previous progress if this is a resumed run.
if ckpt_path.exists():
    records = pd.read_parquet(ckpt_path).to_dict("records")
    done_ids = {r["id"] for r in records}
    print(f"Resuming from checkpoint: {len(done_ids)}/{len(test_df)} rows already done.")
else:
    records = []
    done_ids = set()

start_time = time.time()
stopped_early = False

for row in test_df.itertuples(index=False):
    if row.id in done_ids:
        continue

    pdf_path = CONFIG["test_pdf_dir"] / f"{row.file_id}.pdf"
    top_pages = [int(p) for p in row.top5_pages][:CONFIG["num_pages"]]

    pred = ask_vlm(
        pdf_path=pdf_path,
        file_id=row.file_id,
        page_numbers=top_pages,
        question=row.question,
        answer_format=row.answer_format,
        split="test",
    )

    # Stop cleanly if daily API quota is exhausted instead of saving empty rows.
    if pred["answer"] == "" and ("429" in pred["reasoning"] or "RESOURCE_EXHAUSTED" in pred["reasoning"]):
        print(f"\n[QUOTA HIT at {row.id}] Saving progress and stopping cleanly. Resume later.")
        pd.DataFrame(records).to_parquet(ckpt_path, index=False)
        stopped_early = True
        break

    records.append({
        "id": row.id,
        "answer": pred["answer"],
        "evidence_page_number": str(pred["evidence_page_number"]),
    })
    done_ids.add(row.id)

    # Save checkpoint to Drive every 20 rows.
    if len(records) % 20 == 0 or len(records) == len(test_df):
        pd.DataFrame(records).to_parquet(ckpt_path, index=False)
        elapsed_min = (time.time() - start_time) / 60
        print(f"Saved checkpoint: {len(records)}/{len(test_df)} rows ({elapsed_min:.1f} min elapsed)")

    time.sleep(1)  # Stay cleanly under Gemini's free-tier requests-per-minute limit.

# Export final submission.csv only when all 624 rows are complete.
if not stopped_early and len(records) == len(test_df):
    preds_df = pd.DataFrame(records)
    final_sub = sub_df[["id"]].merge(preds_df, on="id", how="left")
    final_sub.to_csv(sub_path, index=False)

    print("\n=== SUBMISSION READY ===")
    print("Saved to    :", sub_path)
    print("Total rows  :", len(final_sub))
    print("Missing/NaN :", final_sub.isna().sum().to_dict())
    print("Blank ans   :", (final_sub["answer"].astype(str).str.strip() == "").sum())
    print("\nFirst 5 predictions:")
    print(final_sub.head(5))
else:
    print(f"\nPaused at {len(records)}/{len(test_df)} rows. Checkpoint is safely saved on Drive.")

Resuming from checkpoint: 100/624 rows already done.
Saved checkpoint: 120/624 rows (3.1 min elapsed)


KeyboardInterrupt: 

Summary

In [12]:
import pandas as pd

# --- 1. Build clean summary table from Cell 4 ---
val_df = pd.DataFrame(train_results)
val_df["ans_match"] = val_df["true_answer"] == val_df["pred_answer"]
val_df["page_match"] = val_df["true_pages"] == val_df["pred_pages"]

# Save validation results to Drive so Cell 4 has a permanent output file
val_csv_path = CONFIG["output_dir"] / "train_validation_results.csv"
val_df.to_csv(val_csv_path, index=False)

# --- 2. Print end-to-end summary (Cells 1-4) ---
print("=" * 60)
print("DOCULENS - NOTEBOOK 4 SUMMARY (CELLS 1-4)")
print("=" * 60)
print(f"[Cell 1] Indexed Pages : {len(pages_df)} | Train: {len(train_df)} | Test: {len(test_df)}")
print(f"[Cell 2] VLM Backend   : {CONFIG['vlm_model']} (Gemini API)")
print(f"[Cell 3] VLM Pipeline  : Text cache ({len(PAGE_TEXT)} pages) + {CONFIG['render_dpi']} DPI images")
print(f"[Cell 4] Answer Match  : {val_df['ans_match'].sum()}/{len(val_df)} ({val_df['ans_match'].mean() * 100:.1f}%)")
print(f"[Cell 4] Page Match    : {val_df['page_match'].sum()}/{len(val_df)} ({val_df['page_match'].mean() * 100:.1f}%)")
print(f"[Saved ] Validation CSV: {val_csv_path}")
print("=" * 60)

# --- 3. Show the missed train rows for error analysis ---
misses = val_df[~val_df["ans_match"]][["id", "true_answer", "pred_answer", "true_pages", "pred_pages"]]
print(f"\nMissed Questions ({len(misses)}/{len(val_df)}):")
display(misses)

DOCULENS - NOTEBOOK 4 SUMMARY (CELLS 1-4)
[Cell 1] Indexed Pages : 4772 | Train: 16 | Test: 624
[Cell 2] VLM Backend   : gemini-3.5-flash-lite (Gemini API)
[Cell 3] VLM Pipeline  : Text cache (4772 pages) + 150 DPI images
[Cell 4] Answer Match  : 12/16 (75.0%)
[Cell 4] Page Match    : 11/16 (68.8%)
[Saved ] Validation CSV: /content/drive/MyDrive/doculens_lava/outputs/train_validation_results.csv

Missed Questions (4/16):


,id,true_answer,pred_answer,true_pages,pred_pages
3,q_0003,"['ITリテラシーの向上', '実践的なITスキルの習得']","['IT企業への就職・起業', '実践的なITスキルの習得']","[13, 21]","[13, 21]"
9,q_0009,"['4NPhA', '4N1NPT', '24DN1NPT']","['4NPhA', '4NINPT', '24DN1NPT']",[5],[5]
13,q_0013,12.2倍,該当なし,[6],[14]
15,q_0015,"[5411, 5811, 3009]","['5411', '5811', '7011']","[1, 2, 3, 15]","[2, 3, 8]"
